# 🎓 Course PDF Assistant: RAG-based QA System

This notebook demonstrates a **Retrieval-Augmented Generation (RAG)** pipeline designed to answer questions based on a collection of course PDFs.

### 🚀 System Architecture & Pipeline
1. **Environment Setup & GPU Check**: Installs necessary libraries (`pymupdf`, `sentence-transformers`, `faiss-cpu`, `transformers`, `gradio`) and verifies GPU acceleration.
2. **Document Loading**: Loads PDF files using `PyMuPDF` (`fitz`) and extracts text page-by-page.
3. **Text Chunking**: Splits large pages into smaller, overlapping passages (chunks) to preserve local context and fit transformer token limits.
4. **Vector Embedding & Indexing**: Uses a multilingual pre-trained model (`paraphrase-multilingual-MiniLM-L12-v2`) to encode chunks and stores them in a fast **FAISS index** for dense retrieval.
5. **Retrieval**: Finds the top-$k$ most semantically relevant chunks for any user question.
6. **Generation**: Feeds the retrieved passages along with a system prompt to the **Qwen2.5-3B-Instruct** LLM to generate precise, grounded answers with source citations.
7. **User Interface**: Launches an interactive **Gradio** web UI directly from the notebook.

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu accelerate gradio

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU available: True
Tesla T4


## 📂 Step 1: Document Discovery
In this step, we scan our `/content/documents` directory to find and list all available course lecture PDF documents.

In [ ]:
from pathlib import Path

pdf_folder = Path("/content/documents")
pdf_folder.mkdir(exist_ok=True)

pdf_files = sorted(pdf_folder.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s):")
for p in pdf_files:
    print(" -", p.name)

Found 9 PDF(s):
 - Activation Functions.pdf
 - CNNs 2026.pdf
 - Forward and Backward Propagation.pdf
 - Layer Neural Network for XOR.pdf
 - Loss Functions.pdf
 - Perceptron and multi-layer perceptron (MLP).pdf
 - RNN_LSTM_GRU 2026.pdf
 - Training Neural Networks.pdf
 - VAE 2026 ing4.pdf


## 📄 Step 2: Document Processing & Text Extraction
We read each PDF page-by-page. For each page, we extract raw text, track its page number, and keep metadata of its source file so we can cite it later in the generation phase.

In [ ]:
import fitz

def extract_pdf_pages(pdf_path):
    pages, empty = [], 0
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({"text": text, "source": Path(pdf_path).name, "page": page_number})
            else:
                empty += 1
    print(f"{Path(pdf_path).name}: {len(pages)} pages with text, {empty} empty")
    return pages

all_pages = []
for pdf_path in pdf_files:
    all_pages.extend(extract_pdf_pages(pdf_path))

print("\nTotal pages with text:", len(all_pages))

Activation Functions.pdf: 34 pages with text, 0 empty


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


CNNs 2026.pdf: 205 pages with text, 2 empty
Forward and Backward Propagation.pdf: 19 pages with text, 0 empty
Layer Neural Network for XOR.pdf: 10 pages with text, 0 empty
Loss Functions.pdf: 37 pages with text, 0 empty
Perceptron and multi-layer perceptron (MLP).pdf: 222 pages with text, 0 empty
RNN_LSTM_GRU 2026.pdf: 99 pages with text, 0 empty
Training Neural Networks.pdf: 114 pages with text, 0 empty
VAE 2026 ing4.pdf: 77 pages with text, 10 empty

Total pages with text: 817


## ✂️ Step 3: Text Chunking
Because entire pages can sometimes be too large or contain multiple distinct topics, we split our text into smaller windows of 900 characters with a 150-character overlap. This overlap ensures context is not abruptly cut off at chunk boundaries.

In [ ]:
def split_into_chunks(text, chunk_size=900, overlap=150):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = end - overlap
    return chunks

all_chunks = []
for page in all_pages:
    for chunk_id, chunk_text in enumerate(split_into_chunks(page["text"], 900, 150)):
        all_chunks.append({
            "id": len(all_chunks),     # global id = position in FAISS
            "text": chunk_text,
            "source": page["source"],
            "page": page["page"],
            "chunk_id": chunk_id
        })

print("Total chunks:", len(all_chunks))
for c in all_chunks[:3]:
    print(f"\n--- Chunk {c['id']} ({c['source']}, p.{c['page']}) ---\n{c['text']}")

Total chunks: 866

--- Chunk 0 (Activation Functions.pdf, p.1) ---
Chapter 2: 
Feedforward 
Neural 
Networks

--- Chunk 1 (Activation Functions.pdf, p.2) ---
Chapter 2: Feedforward Neural Networks
1. Fundamentals of Neural Networks
1.2. Activation functions: sigmoid, tanh, ReLU, 
Leaky ReLU

--- Chunk 2 (Activation Functions.pdf, p.3) ---
What is an Activation Function?
Definition:
An activation function introduces non-linearity into a neural network, 
allowing it to learn complex patterns instead of just linear relationships.
Without activation functions:
•
A neural network becomes a linear model
•
Cannot solve problems like XOR, image recognition, language 
modeling
Mathematically:
𝑧= 𝑤𝑇𝑥+ 𝑏, 𝑎= 𝑓𝑧


## 🧠 Step 4: Text Embeddings & FAISS Indexing
Using the `paraphrase-multilingual-MiniLM-L12-v2` bi-encoder, we map each text chunk into a high-dimensional vector representation. We then add these embeddings to a **FAISS Inner Product (flat IP) index** for fast similarity-based vector retrieval.

In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda"
)

texts = [c["text"] for c in all_chunks]
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Embedding shape:", embeddings.shape)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Indexed vectors:", index.ntotal)

# Save (order of chunks.json must match the order of vectors in FAISS)
faiss.write_index(index, "/content/rag_index.faiss")
with open("/content/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedding shape: (866, 384)
Indexed vectors: 866


## 🔍 Step 5: Semantic Search (Retrieval)
Here we test our search retrieval system independently of the LLM. When a question is received, it is encoded and matched against our FAISS index to find the most similar passages.

In [ ]:
def retrieve(question, top_k=4):
    q = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, positions = index.search(q, top_k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos == -1:
            continue
        item = dict(all_chunks[pos])
        item["score"] = float(score)
        results.append(item)
    return results

# Test BEFORE loading the LLM. Replace with a topic from your PDFs.
for r in retrieve("What is the ReLU activation function?"):
    print(f"{r['score']:.3f} | {r['source']} p.{r['page']}\n{r['text'][:200]}\n")

0.720 | Activation Functions.pdf p.15
3. ReLU (Rectified Linear Unit) #1
Definition
The ReLU activation function outputs [0,∞)
Formula:
                  𝑓𝑥= max 0 𝑥
                                                               Range 0 ∞

0.666 | Perceptron and multi-layer perceptron (MLP).pdf p.179
Common Activation Functions
4
3
2
1
0
5
-5
-4
-3
-2
-1
0
1
2
3
4
5
ReLU:
(“Rectified Linear Unit”)

0.645 | Perceptron and multi-layer perceptron (MLP).pdf p.180
We will use this “visual shorthand” from  now 
on
+
+
+
Linear  
Activation
ReLU
Activation
Sigmoid  
Activation

0.602 | Perceptron and multi-layer perceptron (MLP).pdf p.24
Activation Functions: Some Common Choices
26
sigmoid
tanh
ReLU
Leaky ReLU
h
a
h
a
h
a
ReLU and Leaky ReLU
are among the most 
popular ones
Preferred more than 
sigmoid. Helps keep the 
mean of the nex



## 🤖 Step 6: Load Large Language Model (Qwen2.5)
We load **Qwen2.5-3B-Instruct**, a powerful, instruction-tuned lightweight model capable of reasoning and synthesis, and place it on our GPU runtime.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0


## 💬 Step 7: Retrieval-Augmented Generation (RAG) Loop
We define the `ask` function that brings together the search engine and the language model. The LLM is strictly instructed via system prompt to answer **only using the supplied contexts** to minimize hallucinations and cite its source material.

In [ ]:
def ask(question, top_k=4, max_new_tokens=300):
    retrieved = retrieve(question, top_k)
    context = "\n\n".join(
        f"[{i}] Source: {r['source']}, page {r['page']}\n{r['text']}"
        for i, r in enumerate(retrieved, start=1)
    )
    messages = [
        {"role": "system", "content":
            "Answer using only the supplied context. If the context does not "
            "contain the answer, say there is not enough information. "
            "Refer to source numbers when possible."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return answer, retrieved

answer, sources = ask("What is the ReLU activation function?")
print(answer)
print("\nSources:", [(s["source"], s["page"]) for s in sources])

The ReLU (Rectified Linear Unit) activation function outputs [0,∞) and is defined by the formula f(x) = max(0, x). Its derivative is f'(x) = 1 for x > 0 and f'(x) = 0 for x ≤ 0.

Sources: [('Activation Functions.pdf', 15), ('Perceptron and multi-layer perceptron (MLP).pdf', 179), ('Perceptron and multi-layer perceptron (MLP).pdf', 180), ('Perceptron and multi-layer perceptron (MLP).pdf', 24)]


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model_name = (
"sentence-transformers/"
"paraphrase-multilingual-MiniLM-L12-v2"
)
embedding_model = SentenceTransformer(embedding_model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 🎨 Step 8: Build Interactive Gradio Web Interface
Finally, we wrap our system into an interactive UI. Users can type questions, dynamically adjust how many chunks ($k$) are fed to the model, and see both the answer and the exact source materials used.

In [ ]:
import gradio as gr

def answer_ui(question, top_k):
    if not question or not question.strip():
        return "Please type a question.", ""
    answer, sources = ask(question, int(top_k))
    src_text = "\n\n".join(
        f"[{i}] {s['source']}, page {s['page']} (score {s['score']:.2f})\n{s['text'][:400]}..."
        for i, s in enumerate(sources, start=1)
    )
    return answer, src_text

demo = gr.Interface(
    fn=answer_ui,
    inputs=[
        gr.Textbox(label="Your question", lines=2),
        gr.Slider(1, 8, value=4, step=1, label="top_k (number of chunks)")
    ],
    outputs=[
        gr.Textbox(label="Answer", lines=8),
        gr.Textbox(label="Retrieved sources", lines=14)
    ],
    title="Course RAG (v1)"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8b39fc9d84740492c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
